In [2]:
from sts.dio.equity import Ticker
import os
from sts.stsig.ta.market_regime import SQNRegime
from sts.stsig.ta.market_regime import TrendSupportResist
from sts.quant.candle import Candle
import pandas as pd
import numpy as np
from sts.system.accounts.pnl_calculators.pnl_SR_cost import PnLCalculationWithSRCosts
from sts.system.accounts.curves.account_curve import AccountCurve
from sts.strategy.utils import plot_buy_sell_action
from sts.stsig.ta.market_regime import BuySellCycle
from sts.plots.time_series_plots import plot_multi_time_series

* BI and SI are typically moving on the opposite direction, this can be verified by the spearman and pearson correlation which is about -40%
* But both BI and SI can move together up and move together down which typically is a different regime
    * when both BI and SI move up together, which is a typically event spured market
    * when both BI and SI comes down, the market will start settledown, which can be used to judge bottom and top ?
* on other case, BI and SI are move on oppsoite direction driven by the participant position rotation. But such scale is typically with window = 10 for spy. longer is not usefully. 

In [16]:
period = "10y"
sym = "SPY"
ticker = Ticker(sym, data_mode=Ticker.DATAMODE.LOCAL, local_data_folder="../../../data/market_data/")
df = Candle(ticker.history(period=period)).log()
buy_sell_cycle = BuySellCycle(window=10)
buy_sell_cycle.set_data(df)

saved file: ../../../data/market_data/equity/SPY.pkl


In [17]:
df = buy_sell_cycle.df

In [32]:
df["Pred"] = df["Close"].iloc[::-1].rolling(window=15).median().iloc[::-1]

In [33]:
df["DeltaPrice"] = df["Pred"] - df["Close"]

In [34]:
df["TotalValue"] = (df["BuyValue"] + df["SellValue"]).rolling(window=100).median()

In [35]:
df["BI"] = df["BuyValue"] / df["TotalValue"]
df["SI"] = df["SellValue"] / df["TotalValue"]

In [36]:
fig = buy_sell_cycle.plot()
fig.update_layout(width=None)

In [37]:
plot_multi_time_series(
    [df["BI"], df["SI"], df["DeltaPrice"]],
    yn_list=["y1", "y1", "y2"],
    ypos_left=[True, True, False],
)

2018.03 market comes down


In [24]:
df[["BI", "SI", "DeltaPrice"]].corr()

,BI,SI,DeltaPrice
BI,1.000000,0.583066,0.108819
SI,0.583066,1.000000,0.135765
DeltaPrice,0.108819,0.135765,1.000000


In [25]:
df["BIM"] = df["BI"].rolling(window=5).median()
df["SIM"] = df["SI"].rolling(window=5).median()

In [26]:
df["DBI"] = df["BI"] - df["BIM"]
df["DSI"] = df["SI"] - df["SIM"]

In [27]:
df[["DBI", "DSI", "DeltaPrice"]].dropna().corr(method="spearman")

,DBI,DSI,DeltaPrice
DBI,1.000000,-0.405105,0.001272
DSI,-0.405105,1.000000,-0.026919
DeltaPrice,0.001272,-0.026919,1.000000
